# Inteligência Estatística Acadêmica

> **Portfolio pessoal.** Este notebook foi extraído de um projeto real de automação de processos acadêmicos e passou por um processo de anonimização: nomes de instituições, pessoas e caminhos locais foram substituídos por termos genéricos, e todos os outputs de execução (que continham dados reais de alunos/professores) foram removidos. O código e a lógica de negócio foram mantidos como estavam em produção.

Gera indicadores estatísticos e um dashboard executivo em PDF a partir da base consolidada de auditoria pedagógica.

---



# SGA v38 — Ajuste final de visualização do PDF

## Ajustes aplicados

- Página 2 foi reorganizada em duas páginas limpas:
  - `Página 2 - Quantitativos executivos`;
  - `Página 2.1 - Horas demandadas`.
- As tabelas não ficam mais aglomeradas ou sobrepostas.
- Os gráficos de barras exibem valores diretamente nas colunas, evitando leitura pelo eixo Y.
- A página de `RPA sinalizado no Mapa e ainda não efetivado no Registro` agora inclui:
  - resumo por ordem: 1ª, 2ª, ESTENDIDA;
  - horas demandadas pendentes por bloco:
    - 1ª ordem + ESTENDIDA;
    - 2ª ordem + ESTENDIDA;
  - lista detalhada dos cursos e disciplinas pendentes.
- A seção RPA segue a ordem:
  1. pendências Mapa x Registro;
  2. gráficos de horas RPA e top cursos RPA;
  3. valores RPA no final da seção.
- O detalhamento docente por dia mantém a regra correta:
  - docentes não RPA = professores únicos;
  - RPA = quantidade de alocações/disciplinas RPA no dia;
  - qtd_docentes = docentes sem RPA + qtd_rpa_dia.


Execute após o Notebook Principal v38.

## Execução

Execute as células em ordem no JupyterLab.

In [ ]:
# ============================================================
# SGA — Notebook de Inteligência Estatística e Histórica v36
# ============================================================
# Este notebook complementa o notebook principal, usando o Excel consolidado
# e a planilha Registro para criar análises estatísticas, históricas,
# cenários e indicadores de risco operacional.
#
# Narrativa:
# 1. Base analítica
# 2. Estatística operacional
# 3. Pareto e concentração
# 4. Sazonalidade do módulo
# 5. Eficiência docente e risco
# 6. Outliers
# 7. Cenários
# 8. Histórico
# 9. Exportações
# ============================================================

In [ ]:
# ============================================================
# 0. Dependências
# ============================================================
import sys, subprocess, importlib.util, warnings, re, unicodedata, json, math
from pathlib import Path
from datetime import datetime

def pacote_instalado(nome_import: str) -> bool:
    return importlib.util.find_spec(nome_import) is not None

def instalar_pacote(nome_pip: str) -> None:
    print(f"Instalando dependência ausente: {nome_pip}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", nome_pip])

DEPENDENCIAS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "xlsxwriter": "xlsxwriter",
    "plotly": "plotly",
    "IPython": "ipython",
    "nbformat": "nbformat",
}

for import_name, pip_name in DEPENDENCIAS.items():
    if not pacote_instalado(import_name):
        try:
            instalar_pacote(pip_name)
        except Exception as e:
            print(f"Não foi possível instalar automaticamente {pip_name}: {e}")
            print(f"Instale manualmente: pip install {pip_name}")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML, Markdown

def _siap_is_notebook():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        return ip is not None and "IPKernelApp" in ip.config
    except Exception:
        return False

if not _siap_is_notebook():
    def display(*args, **kwargs):
        return None
    go.Figure.show = lambda self, *args, **kwargs: None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

print("Dependências carregadas com sucesso.")

In [ ]:
# ============================================================
# 1. Configurações
# ============================================================
BASE_DIR = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
try:
    _write_test = OUTPUT_DIR / "._siap_write_test"
    _write_test.write_text("ok", encoding="utf-8")
    _write_test.unlink(missing_ok=True)
except Exception:
    OUTPUT_DIR = BASE_DIR / "outputs_inteligencia_v52"
    OUTPUT_DIR.mkdir(exist_ok=True)

# Ajuste manual opcional. Se deixar como None, o notebook tenta localizar automaticamente.
CONSOLIDADO_PATH = None
REGISTRO_PATH = None

# Caso queira forçar um módulo específico, preencha aqui. Caso contrário, usa o módulo do Excel consolidado.
MODULO_ALVO = None
ANO_ALVO = None

# Se True, atualiza/append no arquivo histórico local.
ATUALIZAR_HISTORICO = False

CORES = {
    "planejado": "#2563EB",
    "realizado": "#059669",
    "rpa": "#7C3AED",
    "horas": "#D97706",
    "alerta": "#F97316",
    "critico": "#DC2626",
    "neutro": "#64748B",
    "p1": "#2563EB",
    "p2": "#059669",
    "estendida": "#7C3AED",
    "capacidade": "#D97706",
    "alocado": "#059669",
}

ORDEM_DIAS = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]
ORDEM_PERIODOS = ["1ª ordem + ESTENDIDA", "2ª ordem + ESTENDIDA"]

# ----------------------------------------------------------------
# IDENTIFICAÇÃO EXECUTIVA DAS SAÍDAS
# ----------------------------------------------------------------
# Mesmos campos do notebook principal: alimentam o cabeçalho e o rodapé do
# dashboard, mantendo o padrão corporativo (responsável, cargo, atualização).
RELATORIO_AUTOR_NOME = "Seu Nome"
RELATORIO_AUTOR_CARGO = "Analista de Planejamento Acadêmico"
RELATORIO_AUTOR_AREA = ""                      # Opcional. Ex.: "Diretoria de Ensino"
RELATORIO_ORGANIZACAO = "Instituicao EAD"
RELATORIO_FUSO = "America/Sao_Paulo"

def carimbo_atualizacao(fmt: str = "%d/%m/%Y %H:%M") -> str:
    """Data e hora da atualização, no fuso configurado."""
    try:
        from zoneinfo import ZoneInfo
        return datetime.now(ZoneInfo(RELATORIO_FUSO)).strftime(fmt)
    except Exception:
        return datetime.now().strftime(fmt)

def assinatura_relatorio() -> str:
    """Linha de responsabilidade: nome, cargo e área."""
    partes = [str(p).strip() for p in (RELATORIO_AUTOR_NOME, RELATORIO_AUTOR_CARGO, RELATORIO_AUTOR_AREA) if str(p).strip()]
    return " | ".join(partes)

# ----------------------------------------------------------------
# ESCALA DO MAPA DE CALOR
# ----------------------------------------------------------------
# Com dados de amplitude pequena, uma escala linear em paleta clara deixa todas
# as células no mesmo tom. Aqui os limites vêm dos percentis dos valores > 0 e
# as cores são reposicionadas por potência (gamma < 1), abrindo o contraste na
# faixa onde os dados realmente estão.
HEATMAP_GAMMA = 0.55
HEATMAP_PERCENTIS = (2, 98)
HEATMAP_PALETAS = {
    "planejado": ["#F8FAFC", "#DBEAFE", "#93C5FD", "#3B82F6", "#1D4ED8", "#172554"],
    "realizado": ["#F8FAFC", "#D1FAE5", "#6EE7B7", "#10B981", "#047857", "#064E3B"],
    "horas": ["#F8FAFC", "#FEF3C7", "#FCD34D", "#F59E0B", "#B45309", "#78350F"],
}

def limites_heatmap(valores, ignorar_zeros: bool = True):
    """(vmin, vmax) robustos, ignorando células vazias."""
    arr = np.asarray(valores, dtype=float).ravel()
    arr = arr[np.isfinite(arr)]
    if ignorar_zeros:
        nao_zero = arr[arr > 0]
        if nao_zero.size:
            arr = nao_zero
    if arr.size == 0:
        return 0.0, 1.0
    vmin = float(np.percentile(arr, HEATMAP_PERCENTIS[0]))
    vmax = float(np.percentile(arr, HEATMAP_PERCENTIS[1]))
    if vmax <= vmin:
        vmin, vmax = float(arr.min()), float(arr.max())
    if vmax <= vmin:
        vmax = vmin + 1.0
    return vmin, vmax

def escala_heatmap_gamma(cores=None, gamma: float = None):
    """Colorscale Plotly com realce de potência (equivale ao PowerNorm)."""
    cores = list(cores or HEATMAP_PALETAS["planejado"])
    gamma = HEATMAP_GAMMA if gamma is None else gamma
    n = len(cores)
    escala = []
    for i, cor in enumerate(cores):
        t = i / (n - 1)
        pos = 0.0 if i == 0 else (1.0 if i == n - 1 else round(t ** (1.0 / gamma), 6))
        escala.append([pos, cor])
    return escala

px.defaults.template = "plotly_white"

In [ ]:
# ============================================================
# 2. Funções utilitárias
# ============================================================
def remover_acentos(texto):
    if pd.isna(texto):
        return ""
    return "".join(ch for ch in unicodedata.normalize("NFKD", str(texto)) if not unicodedata.combining(ch))

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    s = remover_acentos(str(texto).strip().upper())
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^A-Z0-9ªº|: \-_/]+", " ", s)
    return s.strip()

def nome_exibicao(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return "Não informado"
    return str(texto).strip().title()

def moeda_br(valor):
    try:
        v = float(valor)
        return f"R$ {v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        return "R$ 0,00"

def inteiro_br(valor, sufixo=""):
    try:
        v = float(valor)
        return f"{v:,.0f}{sufixo}".replace(",", ".")
    except Exception:
        return f"0{sufixo}"

def percentual_br(valor, casas=1):
    try:
        return f"{float(valor):.{casas}f}%".replace(".", ",")
    except Exception:
        return "0,0%"

def num(valor, default=0):
    try:
        if pd.isna(valor):
            return default
        return float(valor)
    except Exception:
        return default

def encontrar_arquivo(raiz: Path, padroes, excluir=None, required_sheets=None):
    excluir = excluir or []
    required_sheets = required_sheets or []
    candidatos = []
    for padrao in padroes:
        candidatos.extend(list(raiz.glob(padrao)))
        candidatos.extend(list((raiz / "outputs").glob(padrao)) if (raiz / "outputs").exists() else [])
        candidatos.extend(list((raiz / "outputs_v52").glob(padrao)) if (raiz / "outputs_v52").exists() else [])
        candidatos.extend(list((raiz / "outputs_v36").glob(padrao)) if (raiz / "outputs_v36").exists() else [])
        candidatos.extend(list((raiz / "outputs_siap").glob(padrao)) if (raiz / "outputs_siap").exists() else [])
    candidatos = [p for p in candidatos if p.is_file()]
    for exc in excluir:
        candidatos = [p for p in candidatos if exc.lower() not in p.name.lower()]
    candidatos = sorted(candidatos, key=lambda p: p.stat().st_mtime, reverse=True)
    if required_sheets:
        validos = []
        for cand in candidatos:
            try:
                sheets = pd.ExcelFile(cand).sheet_names
                if all(s in sheets for s in required_sheets):
                    validos.append(cand)
            except Exception:
                pass
        candidatos = validos
    if not candidatos:
        return None
    return candidatos[0]

def padronizar_colunas(df):
    df = df.copy()
    df.columns = [normalizar_texto(c).lower().replace(" ", "_").replace("%", "perc") for c in df.columns]
    return df

def exibir_titulo(titulo, subtitulo=None):
    html = f"""
    <div style="border-left:7px solid {CORES['horas']}; padding:8px 0 8px 18px; margin:28px 0 16px 0;">
        <h2 style="margin:0; color:#0F172A; font-family:Arial, sans-serif;">{titulo}</h2>
        {f'<p style="margin:6px 0 0 0; color:#475569; font-family:Arial, sans-serif;">{subtitulo}</p>' if subtitulo else ''}
    </div>
    """
    display(HTML(html))

def card_html(titulo, valor, subtitulo="", cor="#2563EB"):
    return f"""
    <div style="
        background:#ffffff; border:1px solid #E2E8F0; border-radius:16px; padding:18px;
        box-shadow:0 6px 18px rgba(15,23,42,.08); min-height:110px;">
        <div style="font-size:13px; font-weight:700; color:#475569; text-transform:uppercase; letter-spacing:.04em;">{titulo}</div>
        <div style="font-size:30px; font-weight:800; color:{cor}; margin-top:8px;">{valor}</div>
        <div style="font-size:13px; color:#64748B; margin-top:4px;">{subtitulo}</div>
    </div>
    """

def exibir_cards(cards, colunas=4):
    grid = " ".join(["1fr"] * colunas)
    html = f"<div style='display:grid; grid-template-columns:{grid}; gap:14px; margin:16px 0 28px 0;'>"
    for c in cards:
        html += card_html(**c)
    html += "</div>"
    display(HTML(html))

def estilo_fig(fig, titulo=None, altura=480):
    fig.update_layout(
        title=dict(text=titulo or fig.layout.title.text, x=0.02, xanchor="left", font=dict(size=21, color="#0F172A")),
        font=dict(size=13, color="#0F172A"),
        height=altura,
        margin=dict(l=60, r=30, t=70, b=60),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig.update_xaxes(showgrid=False, linecolor="#CBD5E1")
    fig.update_yaxes(gridcolor="#E2E8F0", linecolor="#CBD5E1")
    return fig

def salvar_fig_html(fig):
    return fig.to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": False})

def tabela_html(df, max_rows=12):
    if df is None or df.empty:
        return "<p class='muted'>Sem dados para exibir.</p>"
    return df.head(max_rows).to_html(index=False, classes="tbl", border=0)

In [ ]:
# ============================================================
# 3. Localização e leitura das bases
# ============================================================
if CONSOLIDADO_PATH is None:
    CONSOLIDADO_PATH = encontrar_arquivo(
        BASE_DIR,
        ["SGA_Auditoria_Pedagogica*.xlsx", "*Excel_Validado.xlsx"],
        excluir=["Inteligencia", "Histórico", "Historico"],
        required_sheets=["Resumo", "Mapa AO VIVO Usado", "Registro Tratado Usado", "Base Docente Atualizada", "Registro Dup Chave"]
    )

if REGISTRO_PATH is None:
    REGISTRO_PATH = encontrar_arquivo(BASE_DIR, ["*REGISTRO*.xlsx"])

if CONSOLIDADO_PATH is None:
    raise FileNotFoundError("Não encontrei o Excel consolidado do SGA. Execute primeiro o notebook principal ou informe CONSOLIDADO_PATH.")

print(f"Excel consolidado SGA: {CONSOLIDADO_PATH}")
print(f"Registro bruto: {REGISTRO_PATH if REGISTRO_PATH else 'não localizado; análises detalhadas do Registro serão limitadas'}")

xls_siap = pd.ExcelFile(CONSOLIDADO_PATH)
abas = xls_siap.sheet_names
print("Abas disponíveis no consolidado:", abas)

def ler_aba(nome, obrigatoria=False):
    if nome in abas:
        return pd.read_excel(CONSOLIDADO_PATH, sheet_name=nome)
    if obrigatoria:
        raise ValueError(f"Aba obrigatória não encontrada: {nome}")
    return pd.DataFrame()

resumo = ler_aba("Resumo", obrigatoria=True)
planejado_ordem = ler_aba("Planejado Ordem")
horas_planejado = ler_aba("Horas Planejado")
mapa_ao_vivo = ler_aba("Mapa AO VIVO Usado")
mapa_chaves = ler_aba("Mapa Chaves Planejado")
registro_tratado_usado = ler_aba("Registro Tratado Usado")
registro_dup_chave = ler_aba("Registro Dup Chave")
registro_chave_remov = ler_aba("Registro Chave Remov")
realizado_ordem = ler_aba("Realizado Ordem")
horas_realizado = ler_aba("Horas Realizado")
horas_comparativo = ler_aba("Horas Comparativo")
rpa_comp = ler_aba("RPA Comparativo")
rpa_mapa = ler_aba("RPA Mapa")
rpa_registro = ler_aba("RPA Registro")
conformidade = ler_aba("Conformidade Chaves")
docentes_param = ler_aba("Base Docente Atualizada")
if docentes_param.empty:
    docentes_param = ler_aba("Parametros Docentes")
docentes_ind = ler_aba("Docentes Individual")
docentes_cargo = ler_aba("Docentes Cargo CH")
docentes_grupo = ler_aba("Docentes Grupo CH")
delta_docente = ler_aba("Delta Docente Geral")
docentes_fora = ler_aba("Governanca Docentes")
if docentes_fora.empty:
    docentes_fora = ler_aba("Docentes Fora Ativos")

linha_resumo = resumo.iloc[0].to_dict()
MODULO = int(linha_resumo.get("modulo", MODULO_ALVO or 0))
ANO = int(linha_resumo.get("ano", ANO_ALVO or 0))
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Módulo analisado: {MODULO} | Ano: {ANO}")

PERIODO_LABEL_MAP = {
    "Período 1": "1ª ordem + ESTENDIDA",
    "Período 2": "2ª ordem + ESTENDIDA",
}

def aplicar_labels_periodos(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df, pd.DataFrame) and not df.empty and "periodo" in df.columns:
        df = df.copy()
        df["periodo"] = df["periodo"].replace(PERIODO_LABEL_MAP)
    return df

for _nm in [
    "horas_planejado", "horas_realizado", "horas_comparativo",
    "delta_docente", "docentes_ind", "docentes_cargo", "docentes_grupo"
]:
    if _nm in globals():
        globals()[_nm] = aplicar_labels_periodos(globals()[_nm])

In [ ]:
# ============================================================
# 4. Leitura detalhada do Registro bruto
# ============================================================
def detectar_linha_cabecalho_registro(path, sheet="PREENCHIMENTO|LETIVAS"):
    raw = pd.read_excel(path, sheet_name=sheet, header=None, nrows=30)
    for i, row in raw.iterrows():
        vals = [normalizar_texto(v) for v in row.tolist()]
        texto = " ".join(vals)
        if "PROFESSOR" in texto and "DISCIPLINA" in texto and "HORARIO" in texto:
            return i
    return 8

def normalizar_ordem(x):
    s = normalizar_texto(x)
    if s in ["1", "1.0", "1ª", "1A", "1º"] or s.startswith("1"):
        return "1ª"
    if s in ["2", "2.0", "2ª", "2A", "2º"] or s.startswith("2"):
        return "2ª"
    if "EST" in s:
        return "ESTENDIDA"
    if s in ["", "NAN"]:
        return np.nan
    return s

def extrair_dia(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    # Exemplos: 4 - Quarta, Quinta, 2 - Segunda
    m = re.search(r"([A-Za-zÀ-ÿ]+)\s*$", s)
    if m:
        return m.group(1).capitalize()
    return s.capitalize()

def extrair_modulo_ano(x):
    if pd.isna(x):
        return (np.nan, np.nan)
    s = str(x)
    nums = re.findall(r"\d+", s)
    if len(nums) >= 2:
        return (int(nums[0]), int(nums[1]))
    if len(nums) == 1:
        return (int(nums[0]), np.nan)
    return (np.nan, np.nan)

def ler_registro_detalhado(path, modulo_alvo=None, ano_alvo=None):
    if path is None or not Path(path).exists():
        return pd.DataFrame(), pd.DataFrame()
    header = detectar_linha_cabecalho_registro(path)
    df = pd.read_excel(path, sheet_name="PREENCHIMENTO|LETIVAS", header=header)
    df = df.dropna(how="all").copy()

    # Mapeamento resiliente de colunas
    colmap = {}
    for c in df.columns:
        cn = normalizar_texto(c)
        if "PROFESSOR" in cn: colmap[c] = "professor"
        elif cn == "CHAPA": colmap[c] = "chapa"
        elif "CH SEMANAL" in cn: colmap[c] = "ch_semanal"
        elif "CH LETIVA" in cn: colmap[c] = "ch_letiva"
        elif cn == "CARGO": colmap[c] = "cargo"
        elif "MODULO" in cn: colmap[c] = "modulo_raw"
        elif cn == "ORDEM": colmap[c] = "ordem_raw"
        elif "DIA" in cn and "TRANSMISSAO" in cn: colmap[c] = "dia_raw"
        elif "HORARIO" in cn: colmap[c] = "horario"
        elif "DISCIPLINA" in cn and "NOME" in cn: colmap[c] = "disciplina"
        elif "R" in cn and "RPA" in cn: colmap[c] = "valor_rpa_raw"
        elif "OBSERVACAO" in cn: colmap[c] = "observacao"
    df = df.rename(columns=colmap)
    keep = [c for c in ["professor","chapa","ch_semanal","ch_letiva","cargo","modulo_raw","ordem_raw","dia_raw","horario","disciplina","observacao","valor_rpa_raw"] if c in df.columns]
    df = df[keep].copy()

    for c in ["professor","cargo","disciplina","horario"]:
        if c not in df.columns:
            df[c] = np.nan

    df["professor_key"] = df["professor"].map(normalizar_texto)
    df["disciplina_key"] = df["disciplina"].map(normalizar_texto)
    df["cargo_key"] = df["cargo"].map(normalizar_texto)
    df["ordem"] = df.get("ordem_raw", pd.Series(index=df.index)).map(normalizar_ordem)
    df["dia_semana"] = df.get("dia_raw", pd.Series(index=df.index)).map(extrair_dia)
    df["horario"] = df["horario"].astype(str).str.strip()

    if "modulo_raw" in df.columns:
        mods = df["modulo_raw"].apply(extrair_modulo_ano)
        df["modulo"] = [m[0] for m in mods]
        df["ano"] = [m[1] for m in mods]
    else:
        df["modulo"] = modulo_alvo
        df["ano"] = ano_alvo

    df["ch_semanal"] = pd.to_numeric(df.get("ch_semanal", np.nan), errors="coerce")
    df["ch_letiva"] = pd.to_numeric(df.get("ch_letiva", np.nan), errors="coerce")
    df["valor_rpa"] = pd.to_numeric(
        df.get("valor_rpa_raw", pd.Series(index=df.index)).astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce"
    )

    df = df[df["disciplina_key"].ne("")]
    df = df[df["professor_key"].ne("")]
    df = df[df["ordem"].isin(["1ª", "2ª", "ESTENDIDA"])]
    if modulo_alvo:
        df = df[df["modulo"].fillna(0).astype(int).eq(int(modulo_alvo))]
    if ano_alvo:
        df = df[(df["ano"].isna()) | (df["ano"].fillna(0).astype(int).eq(int(ano_alvo)))]

    # Regra v36: duplicatas são identificadas para governança, mas entram nos cálculos.
    # A qualidade do Registro será corrigida pela área responsável e refletida automaticamente.
    dup_cols = [c for c in ["modulo", "ano", "professor_key", "chapa", "cargo", "ch_semanal", "ch_letiva", "disciplina_key", "ordem", "dia_semana", "horario", "observacao", "valor_rpa_raw"] if c in df.columns]
    chave_dup = df[dup_cols].copy().astype("string").fillna("<NA>").agg("||".join, axis=1)
    df["grupo_duplicata_exata"] = pd.factorize(chave_dup)[0] + 1
    df["duplicata_exata_registro"] = df.duplicated(subset=dup_cols, keep=False)
    df["removida_por_duplicata_exata"] = df.duplicated(subset=dup_cols, keep="first")
    registro_duplicatas_exatas = df[df["duplicata_exata_registro"].eq(True)].copy()
    registro_duplicatas_removidas = df[df["removida_por_duplicata_exata"].eq(True)].copy()
    if not registro_duplicatas_exatas.empty:
        contagem_dup = registro_duplicatas_exatas.groupby("grupo_duplicata_exata").size().rename("qtd_linhas_grupo").reset_index()
        registro_duplicatas_exatas = registro_duplicatas_exatas.merge(contagem_dup, on="grupo_duplicata_exata", how="left")
        registro_duplicatas_removidas = registro_duplicatas_removidas.merge(contagem_dup, on="grupo_duplicata_exata", how="left")
    # v36: não removemos duplicatas exatas do cálculo.

    df["chave_sessao"] = df["disciplina_key"] + "||" + df["ordem"].astype(str) + "||" + df["dia_semana"].astype(str) + "||" + df["horario"].astype(str)
    df["horas_sessao"] = 2

    # v36: todas as linhas válidas do Registro entram nos cálculos; sem deduplicação por sessão.
    sessoes = df.copy()

    # Expansão operacional por período: ESTENDIDA pesa em P1 e P2.
    linhas = []
    for _, r in sessoes.iterrows():
        if r["ordem"] == "1ª":
            periodos = ["1ª ordem + ESTENDIDA"]
        elif r["ordem"] == "2ª":
            periodos = ["2ª ordem + ESTENDIDA"]
        elif r["ordem"] == "ESTENDIDA":
            periodos = ["1ª ordem + ESTENDIDA", "2ª ordem + ESTENDIDA"]
        else:
            periodos = []
        for p in periodos:
            rr = r.to_dict()
            rr["periodo"] = p
            rr["horas_periodo"] = 2
            linhas.append(rr)
    expandido = pd.DataFrame(linhas)
    return sessoes, expandido, registro_duplicatas_exatas, registro_duplicatas_removidas

registro_sessoes, registro_periodos, registro_duplicatas_exatas, registro_duplicatas_removidas = ler_registro_detalhado(REGISTRO_PATH, MODULO, ANO)

print("Sessões/linhas realizadas consideradas:", len(registro_sessoes))
print("Linhas expandidas por período:", len(registro_periodos))
print("Duplicatas exatas identificadas no Registro:", len(registro_duplicatas_removidas))
if not registro_sessoes.empty:
    display(registro_sessoes.head())

In [ ]:
# ============================================================
# 5. Base analítica docente e checks de consistência
# ============================================================
# Base de capacidade docente exportada pelo notebook principal.
# Regra v36: além dos docentes com STATUS = ATIVO na base docente atualizada, docentes alocados no módulo
# que não estão ativos também entram nos cálculos, mantendo-se a tabela de governança para ajuste no RM.
docentes_param = docentes_param.copy()
if not docentes_param.empty:
    docentes_param["docente_ativo"] = docentes_param.get("docente_ativo", False).astype(bool)
    docentes_ativos = docentes_param[docentes_param["docente_ativo"] == True].copy()
else:
    docentes_ativos = pd.DataFrame()

# Base docente individual já calculada pelo notebook principal. Esta é a fonte preferencial para capacidade,
# pois já contempla a regra v36 de inclusão de docentes alocados fora do status ativo.
docentes_ind = docentes_ind.copy()
if not docentes_ind.empty:
    if "incluido_base_capacidade" in docentes_ind.columns:
        docentes_ind_base = docentes_ind[docentes_ind["incluido_base_capacidade"].fillna(False).astype(bool)].copy()
    else:
        docentes_ind_base = docentes_ind[docentes_ind.get("docente_ativo", True) == True].copy() if "docente_ativo" in docentes_ind.columns else docentes_ind.copy()
else:
    docentes_ind_base = pd.DataFrame()

# Mantém compatibilidade com nomes usados nas seções antigas.
docentes_ind_ativos = docentes_ind_base.copy()

# Alocação por docente a partir do Registro integral, para análises estatísticas.
if not registro_periodos.empty:
    aloc_doc = (
        registro_periodos
        .groupby(["professor_key","professor","cargo"], dropna=False)
        .agg(
            horas_total=("horas_periodo","sum"),
            horas_p1=("horas_periodo", lambda s: s[registro_periodos.loc[s.index, "periodo"].eq("1ª ordem + ESTENDIDA")].sum()),
            horas_p2=("horas_periodo", lambda s: s[registro_periodos.loc[s.index, "periodo"].eq("2ª ordem + ESTENDIDA")].sum()),
            disciplinas_distintas=("disciplina_key","nunique"),
            sessoes=("chave_sessao","nunique"),
            dias_distintos=("dia_semana","nunique"),
            horarios_distintos=("horario","nunique"),
            valor_rpa=("valor_rpa","sum"),
            rpa_linhas=("valor_rpa", lambda s: s.notna().sum()),
        ).reset_index()
    )
else:
    aloc_doc = pd.DataFrame()

# Enriquecimento com capacidade da base final do Notebook Principal.
# Usa docentes_ind_base, que inclui ativos e, pela regra v36, docentes alocados fora do ativo.
cap_cols = ["professor_key","cargo","cargo_key","ch_semanal","ch_letiva","ch_letiva_alocavel","capacidade_periodo","capacidade_modulo_2_periodos","docente_ativo","incluido_por_alocacao","incluido_base_capacidade","origem_capacidade"]
cap_cols = [c for c in cap_cols if c in docentes_ind_base.columns]
cap_base = docentes_ind_base[cap_cols].drop_duplicates("professor_key") if not docentes_ind_base.empty else pd.DataFrame()

if not aloc_doc.empty and not cap_base.empty:
    aloc_doc = aloc_doc.merge(cap_base, on="professor_key", how="left", suffixes=("_registro","_parametros"))
    aloc_doc["docente_ativo_base"] = aloc_doc["docente_ativo"].fillna(False)
    aloc_doc["cargo_final"] = aloc_doc["cargo_parametros"].fillna(aloc_doc["cargo_registro"] if "cargo_registro" in aloc_doc.columns else aloc_doc.get("cargo"))
    aloc_doc["ch_letiva_alocavel"] = pd.to_numeric(aloc_doc.get("ch_letiva_alocavel", 0), errors="coerce").fillna(0)
    aloc_doc["utilizacao_p1_%"] = np.where(aloc_doc["ch_letiva_alocavel"] > 0, aloc_doc["horas_p1"] / aloc_doc["ch_letiva_alocavel"] * 100, np.nan)
    aloc_doc["utilizacao_p2_%"] = np.where(aloc_doc["ch_letiva_alocavel"] > 0, aloc_doc["horas_p2"] / aloc_doc["ch_letiva_alocavel"] * 100, np.nan)
    aloc_doc["utilizacao_max_%"] = aloc_doc[["utilizacao_p1_%","utilizacao_p2_%"]].max(axis=1)
else:
    aloc_doc["docente_ativo_base"] = False

# Checks de consistência com o notebook principal
check = pd.DataFrame()
if not horas_realizado.empty and not registro_periodos.empty:
    calc = registro_periodos.groupby("periodo")["horas_periodo"].sum().reindex(ORDEM_PERIODOS).reset_index()
    calc.columns = ["periodo","horas_registro_recalculado"]
    check = horas_realizado[["periodo","horas"]].merge(calc, on="periodo", how="outer")
    check = check.rename(columns={"horas":"horas_ato2_siap"})
    check["delta"] = check["horas_registro_recalculado"] - check["horas_ato2_siap"]
    display(check)

In [ ]:
# ============================================================
# 5B. Planejado x Realizado por dimensões
# ============================================================
exibir_titulo(
    "1B. Planejado × Realizado por dimensões",
    "Todas as análises estratégicas passam a ter contraponto entre Mapa e Registro para aproximar a leitura da verdade operacional."
)

def expandir_periodos_ordem(df: pd.DataFrame, fonte: str = "Planejado") -> pd.DataFrame:
    """Expande linhas por período operacional: 1ª->P1, 2ª->P2, ESTENDIDA->P1 e P2."""
    if df is None or df.empty or "ordem" not in df.columns:
        return pd.DataFrame()
    linhas = []
    d = df.copy()
    for _, r in d.iterrows():
        ordem = str(r.get("ordem", "")).strip()
        if ordem == "1ª":
            periodos = ["1ª ordem + ESTENDIDA"]
        elif ordem == "2ª":
            periodos = ["2ª ordem + ESTENDIDA"]
        elif ordem == "ESTENDIDA":
            periodos = ["1ª ordem + ESTENDIDA", "2ª ordem + ESTENDIDA"]
        else:
            periodos = []
        for periodo in periodos:
            rr = r.to_dict()
            rr["periodo"] = periodo
            rr["horas_periodo"] = 2
            rr["origem"] = fonte
            linhas.append(rr)
    return pd.DataFrame(linhas)

def preparar_realizado_periodos_unico() -> pd.DataFrame:
    """Usa a base do Registro já deduplicada pelo notebook principal; se indisponível, usa registro_periodos."""
    if not registro_tratado_usado.empty:
        base_real = registro_tratado_usado.copy()
        # blindagem: se a aba vier de versão anterior, remove duplicidades por chave operacional aqui também
        key_cols = [c for c in ["disciplina_key", "ordem", "dia_semana", "horario"] if c in base_real.columns]
        if key_cols:
            base_real = base_real.sort_values([c for c in ["linha_origem"] if c in base_real.columns]).drop_duplicates(key_cols, keep="first")
        if "chave_conformidade" not in base_real.columns:
            base_real["chave_conformidade"] = (
                base_real.get("disciplina_key", "").astype(str) + "||" +
                base_real.get("ordem", "").astype(str) + "||" +
                base_real.get("dia_semana", "").astype(str) + "||" +
                base_real.get("horario", "").astype(str)
            )
        return expandir_periodos_ordem(base_real, fonte="Realizado")
    if not registro_periodos.empty:
        base_real = registro_periodos.copy()
        if "chave_conformidade" not in base_real.columns:
            if "chave_sessao" in base_real.columns:
                base_real["chave_conformidade"] = base_real["chave_sessao"]
            else:
                base_real["chave_conformidade"] = (
                    base_real.get("disciplina_key", "").astype(str) + "||" +
                    base_real.get("ordem", "").astype(str) + "||" +
                    base_real.get("dia_semana", "").astype(str) + "||" +
                    base_real.get("horario", "").astype(str)
                )
        base_real["origem"] = "Realizado"
        return base_real
    return pd.DataFrame()

def comparar_planejado_realizado(planejado_df: pd.DataFrame, realizado_df: pd.DataFrame, grupos: list) -> pd.DataFrame:
    if planejado_df is None or planejado_df.empty:
        p = pd.DataFrame(columns=grupos + ["horas_planejado"])
    else:
        p = planejado_df.groupby(grupos, dropna=False)["horas_periodo"].sum().reset_index(name="horas_planejado")
    if realizado_df is None or realizado_df.empty:
        r = pd.DataFrame(columns=grupos + ["horas_realizado"])
    else:
        r = realizado_df.groupby(grupos, dropna=False)["horas_periodo"].sum().reset_index(name="horas_realizado")
    comp = p.merge(r, on=grupos, how="outer").fillna(0)
    comp["delta_horas"] = comp["horas_realizado"] - comp["horas_planejado"]
    comp["aderencia_%"] = np.where(comp["horas_planejado"].gt(0), comp["horas_realizado"] / comp["horas_planejado"] * 100, np.nan)
    return comp

# Base física de aulas planejadas: chaves deduplicadas do Mapa.
planejado_periodos = expandir_periodos_ordem(mapa_chaves, fonte="Planejado")

# Base oficial do realizado: Registro tratado/deduplicado, alinhado ao Ato 2.
realizado_periodos_unico = preparar_realizado_periodos_unico()

# Comparativos físicos por dimensão operacional.
comp_ordem_pr = comparar_planejado_realizado(planejado_periodos, realizado_periodos_unico, ["ordem"])
comp_dia_periodo_pr = comparar_planejado_realizado(planejado_periodos, realizado_periodos_unico, ["periodo", "dia_semana"])
comp_horario_periodo_pr = comparar_planejado_realizado(planejado_periodos, realizado_periodos_unico, ["periodo", "horario"])

# Comparativo por curso: usa as linhas do Mapa AO VIVO por curso.
# Observação: quando uma aula é sinérgica, a hora realizada é atribuída aos cursos vinculados àquela chave,
# permitindo enxergar cobertura por curso atendido, e não apenas sessão física.
planejado_curso_periodos = expandir_periodos_ordem(mapa_ao_vivo, fonte="Planejado") if not mapa_ao_vivo.empty else pd.DataFrame()
realizado_curso_periodos = pd.DataFrame()
if not realizado_periodos_unico.empty and not mapa_ao_vivo.empty and "chave_conformidade" in mapa_ao_vivo.columns:
    mapa_cursos = mapa_ao_vivo[["chave_conformidade", "curso_nome", "curso_codigo"]].drop_duplicates()
    realizado_curso_periodos = realizado_periodos_unico.merge(mapa_cursos, on="chave_conformidade", how="left")
    realizado_curso_periodos["curso_nome"] = realizado_curso_periodos["curso_nome"].fillna("SEM CORRESPONDÊNCIA NO MAPA")
    realizado_curso_periodos["curso_codigo"] = realizado_curso_periodos["curso_codigo"].fillna("SEM_MAPA")

comp_curso_periodo_pr = comparar_planejado_realizado(planejado_curso_periodos, realizado_curso_periodos, ["periodo", "curso_nome"])

# ------------------------------------------------------------------
# Regra v13 — demanda semanal por curso
# ------------------------------------------------------------------
# A tabela/gráfico de cursos representa demanda semanal por período, não carga acumulada do módulo.
# Justificativa da regra:
# - Disciplina 1ª ou 2ª: 8h em 4 semanas => 2h por semana;
# - Disciplina ESTENDIDA: 18h em 9 semanas => 2h por semana.
# Portanto:
#   P1 = (qtd 1ª + qtd ESTENDIDA) × 2h/semana
#   P2 = (qtd 2ª + qtd ESTENDIDA) × 2h/semana
curso_semanal_periodo_pr = comp_curso_periodo_pr.copy()
if not curso_semanal_periodo_pr.empty:
    curso_semanal_periodo_pr = curso_semanal_periodo_pr.rename(columns={
        "horas_planejado": "demanda_semanal_planejada_horas",
        "horas_realizado": "demanda_semanal_realizada_horas",
    })
    curso_semanal_periodo_pr["delta_semanal_horas"] = (
        curso_semanal_periodo_pr["demanda_semanal_realizada_horas"] -
        curso_semanal_periodo_pr["demanda_semanal_planejada_horas"]
    )
    curso_semanal_periodo_pr = curso_semanal_periodo_pr.drop(columns=["delta_horas"], errors="ignore")
    curso_semanal_periodo_pr["aderencia_%"] = np.where(
        curso_semanal_periodo_pr["demanda_semanal_planejada_horas"].gt(0),
        curso_semanal_periodo_pr["demanda_semanal_realizada_horas"] /
        curso_semanal_periodo_pr["demanda_semanal_planejada_horas"] * 100,
        np.nan
    )
    curso_semanal_periodo_pr["composicao_periodo"] = curso_semanal_periodo_pr["periodo"].map({
        "1ª ordem + ESTENDIDA": "1ª + ESTENDIDA",
        "2ª ordem + ESTENDIDA": "2ª + ESTENDIDA",
    }).fillna(curso_semanal_periodo_pr["periodo"].astype(str))
    curso_semanal_periodo_pr["regra_calculo"] = (
        "Demanda semanal = disciplinas ativas no período × 2h/semana. "
        "1ª/2ª duram 4 semanas (8h); ESTENDIDA dura 9 semanas (18h)."
    )

if not curso_semanal_periodo_pr.empty:
    curso_semanal_rank_pr = (
        curso_semanal_periodo_pr
        .assign(horas_referencia=lambda d: d[["demanda_semanal_planejada_horas", "demanda_semanal_realizada_horas"]].max(axis=1))
        .sort_values("horas_referencia", ascending=False)
    )
else:
    curso_semanal_rank_pr = pd.DataFrame()

# Apoio legado: total por curso, sem uso como gráfico principal.
comp_curso_total_pr = comparar_planejado_realizado(planejado_curso_periodos, realizado_curso_periodos, ["curso_nome"])
if not comp_curso_total_pr.empty:
    comp_curso_total_pr["horas_referencia"] = comp_curso_total_pr[["horas_planejado", "horas_realizado"]].max(axis=1)
    comp_curso_total_pr = comp_curso_total_pr.sort_values("horas_referencia", ascending=False)

# Figuras principais do contraponto.
if not comp_ordem_pr.empty:
    ordem_plot = comp_ordem_pr.melt(id_vars=["ordem"], value_vars=["horas_planejado", "horas_realizado"], var_name="origem", value_name="horas")
    ordem_plot["origem"] = ordem_plot["origem"].map({"horas_planejado": "Planejado", "horas_realizado": "Realizado"})
    fig_ordem_pr = px.bar(
        ordem_plot,
        x="ordem", y="horas", color="origem", barmode="group", text="horas",
        color_discrete_map={"Planejado": CORES["planejado"], "Realizado": CORES["realizado"]},
        labels={"ordem": "Ordem", "horas": "Horas", "origem": "Origem"},
    )
    fig_ordem_pr.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
    estilo_fig(fig_ordem_pr, "Planejado × realizado por ordem", 460).show()

if not comp_dia_periodo_pr.empty:
    dia_plot = comp_dia_periodo_pr.copy()
    dia_plot["dia_semana"] = pd.Categorical(dia_plot["dia_semana"], ORDEM_DIAS, ordered=True)
    dia_plot = dia_plot.sort_values(["periodo", "dia_semana"])
    dia_plot_long = dia_plot.melt(id_vars=["periodo", "dia_semana"], value_vars=["horas_planejado", "horas_realizado"], var_name="origem", value_name="horas")
    dia_plot_long["origem"] = dia_plot_long["origem"].map({"horas_planejado": "Planejado", "horas_realizado": "Realizado"})
    fig_dia_pr = px.bar(
        dia_plot_long, x="dia_semana", y="horas", color="origem", facet_col="periodo",
        barmode="group", text="horas",
        color_discrete_map={"Planejado": CORES["planejado"], "Realizado": CORES["realizado"]},
        category_orders={"dia_semana": ORDEM_DIAS, "periodo": ORDEM_PERIODOS},
        labels={"dia_semana": "Dia da semana", "horas": "Horas", "origem": "Origem"},
    )
    fig_dia_pr.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
    estilo_fig(fig_dia_pr, "Dia da semana — Planejado × Realizado por período", 560).show()

if not comp_horario_periodo_pr.empty:
    horario_plot = comp_horario_periodo_pr.copy()
    horario_plot_long = horario_plot.melt(id_vars=["periodo", "horario"], value_vars=["horas_planejado", "horas_realizado"], var_name="origem", value_name="horas")
    horario_plot_long["origem"] = horario_plot_long["origem"].map({"horas_planejado": "Planejado", "horas_realizado": "Realizado"})
    fig_horario_pr = px.bar(
        horario_plot_long, x="horario", y="horas", color="origem", facet_col="periodo",
        barmode="group", text="horas",
        color_discrete_map={"Planejado": CORES["planejado"], "Realizado": CORES["realizado"]},
        category_orders={"periodo": ORDEM_PERIODOS},
        labels={"horario": "Horário", "horas": "Horas", "origem": "Origem"},
    )
    fig_horario_pr.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
    estilo_fig(fig_horario_pr, "Horário — Planejado × Realizado por período", 560).show()

if 'curso_semanal_rank_pr' in globals() and not curso_semanal_rank_pr.empty:
    top_cursos_lista = curso_semanal_rank_pr["curso_nome"].drop_duplicates().head(18).tolist()
    curso_plot_base = curso_semanal_periodo_pr[curso_semanal_periodo_pr["curso_nome"].isin(top_cursos_lista)].copy()
    curso_plot_base["periodo"] = pd.Categorical(curso_plot_base["periodo"], ORDEM_PERIODOS, ordered=True)
    curso_plot_base = curso_plot_base.sort_values(["periodo", "demanda_semanal_planejada_horas"], ascending=[True, True])
    curso_plot = curso_plot_base.melt(
        id_vars=["periodo", "composicao_periodo", "curso_nome"],
        value_vars=["demanda_semanal_planejada_horas", "demanda_semanal_realizada_horas"],
        var_name="origem",
        value_name="horas_semanais",
    )
    curso_plot["origem"] = curso_plot["origem"].map({
        "demanda_semanal_planejada_horas": "Planejado",
        "demanda_semanal_realizada_horas": "Realizado",
    })
    fig_curso_pr = px.bar(
        curso_plot,
        x="horas_semanais",
        y="curso_nome",
        color="origem",
        facet_col="periodo",
        orientation="h",
        barmode="group",
        text="horas_semanais",
        color_discrete_map={"Planejado": CORES["planejado"], "Realizado": CORES["realizado"]},
        category_orders={"periodo": ORDEM_PERIODOS},
        labels={
            "curso_nome": "Curso",
            "horas_semanais": "Horas semanais",
            "origem": "Origem",
            "periodo": "Período",
        },
    )
    fig_curso_pr.update_traces(texttemplate="%{text:.0f}h/sem", textposition="outside")
    estilo_fig(fig_curso_pr, "Demanda semanal por curso — Planejado × Realizado", 820).show()

    display(Markdown("""
**Regra da tabela de cursos:** disciplinas de 1ª e 2ª ordem duram 4 semanas, totalizando 8h por disciplina; disciplinas ESTENDIDAS duram 9 semanas, totalizando 18h.  
Como todas equivalem a 2h por semana, a demanda semanal por curso é calculada como:
- **1ª ordem + ESTENDIDA:** (disciplinas 1ª + ESTENDIDA) × 2h/semana;
- **2ª ordem + ESTENDIDA:** (disciplinas 2ª + ESTENDIDA) × 2h/semana.
"""))
    display(curso_semanal_rank_pr[[
        "periodo", "composicao_periodo", "curso_nome",
        "demanda_semanal_planejada_horas", "demanda_semanal_realizada_horas",
        "delta_semanal_horas", "aderencia_%"
    ]].head(30))

display(Markdown("""
**Regra de leitura:** a partir desta versão, as análises de horas são apresentadas com contraponto entre Planejado e Realizado sempre que a dimensão existe nas duas bases.  
Para cursos, a leitura usa o vínculo do Mapa: quando uma aula realizada atende cursos sinérgicos, a hora realizada é atribuída aos cursos associados à chave da aula.
"""))

In [ ]:
# ============================================================
# 6. Abertura executiva
# ============================================================
exibir_titulo(
    "Ato 6 — Inteligência Estatística e Histórica",
    "Análises estatísticas, concentração, risco operacional, cenários e histórico a partir do Registro estruturado e do consolidado SGA."
)

cards = [
    {"titulo":"Módulo", "valor": f"{MODULO}/{ANO}", "subtitulo":"base analisada", "cor": CORES["neutro"]},
    {"titulo":"Sessões realizadas", "valor": inteiro_br(len(registro_sessoes)), "subtitulo":"Registro integral", "cor": CORES["realizado"]},
    {"titulo":"Horas P1 realizadas", "valor": inteiro_br(horas_realizado.loc[horas_realizado['periodo'].eq('1ª ordem + ESTENDIDA'), 'horas'].sum(), "h"), "subtitulo":"1ª + ESTENDIDA", "cor": CORES["p1"]},
    {"titulo":"Horas P2 realizadas", "valor": inteiro_br(horas_realizado.loc[horas_realizado['periodo'].eq('2ª ordem + ESTENDIDA'), 'horas'].sum(), "h"), "subtitulo":"2ª + ESTENDIDA", "cor": CORES["p2"]},
    {"titulo":"Base capacidade", "valor": inteiro_br(linha_resumo.get("professores_unicos_base_capacidade", linha_resumo.get("professores_unicos_ativos_parametros", 0))), "subtitulo":"ativos + alocados regra v36", "cor": CORES["capacidade"]},
    {"titulo":"CH alocável", "valor": inteiro_br(linha_resumo.get("ch_letiva_alocavel_total", 0), "h"), "subtitulo":"por período", "cor": CORES["horas"]},
    {"titulo":"RPA realizado", "valor": moeda_br(linha_resumo.get("valor_rpa_registro", 0)), "subtitulo": f"{inteiro_br(linha_resumo.get('rpa_registro_total',0))} RPAs no Registro", "cor": CORES["rpa"]},
    {"titulo":"Conformidade", "valor": percentual_br(linha_resumo.get("conformidade_%",0), 1), "subtitulo":"Mapa × Registro", "cor": CORES["alerta"]},
]
exibir_cards(cards, colunas=4)

In [ ]:
# ============================================================
# 7. Estatística operacional do Registro por período
# ============================================================
exibir_titulo(
    "1. Estatística operacional do Registro por período",
    "Sempre que a análise envolve horas, a leitura é aberta por 1ª ordem + ESTENDIDA = 1ª + ESTENDIDA e 2ª ordem + ESTENDIDA = 2ª + ESTENDIDA. Isso evita misturar cargas com regras operacionais diferentes."
)

if not registro_periodos.empty:
    # ------------------------------------------------------------------
    # Distribuição por dia da semana, sempre aberta por período.
    # ------------------------------------------------------------------
    dist_dia = (
        registro_periodos.groupby(["periodo", "dia_semana"], dropna=False)
        .agg(sessoes=("chave_sessao", "nunique"), horas=("horas_periodo", "sum"))
        .reset_index()
    )
    dist_dia["periodo"] = pd.Categorical(dist_dia["periodo"], ORDEM_PERIODOS, ordered=True)
    dist_dia["dia_semana"] = pd.Categorical(dist_dia["dia_semana"], ORDEM_DIAS, ordered=True)
    dist_dia = dist_dia.sort_values(["periodo", "dia_semana"])
    dist_dia["dia_periodo"] = dist_dia["periodo"].astype(str) + " | " + dist_dia["dia_semana"].astype(str)

    fig_dia = px.bar(
        dist_dia,
        x="dia_semana",
        y="horas",
        color="periodo",
        barmode="group",
        text="horas",
        color_discrete_map={"1ª ordem + ESTENDIDA": CORES["p1"], "2ª ordem + ESTENDIDA": CORES["p2"]},
        category_orders={"dia_semana": ORDEM_DIAS, "periodo": ORDEM_PERIODOS},
        labels={
            "dia_semana": "Dia da semana",
            "horas": "Horas",
            "periodo": "Período operacional",
        },
    )
    fig_dia.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
    estilo_fig(fig_dia, "Carga realizada por dia — P1 (1ª + ESTENDIDA) × P2 (2ª + ESTENDIDA)", 500).show()

    # ------------------------------------------------------------------
    # Distribuição por horário, sempre aberta por período.
    # ------------------------------------------------------------------
    dist_horario = (
        registro_periodos.groupby(["periodo", "horario"], dropna=False)
        .agg(sessoes=("chave_sessao", "nunique"), horas=("horas_periodo", "sum"))
        .reset_index()
    )
    total_horario = dist_horario.groupby("horario", dropna=False)["horas"].sum().sort_values(ascending=False)
    top_horarios = total_horario.head(15).index.tolist()
    dist_horario = dist_horario[dist_horario["horario"].isin(top_horarios)].copy()
    dist_horario["periodo"] = pd.Categorical(dist_horario["periodo"], ORDEM_PERIODOS, ordered=True)
    dist_horario["horario"] = pd.Categorical(dist_horario["horario"], list(reversed(top_horarios)), ordered=True)
    dist_horario = dist_horario.sort_values(["horario", "periodo"])
    dist_horario["horario_periodo"] = dist_horario["periodo"].astype(str) + " | " + dist_horario["horario"].astype(str)

    fig_horario = px.bar(
        dist_horario,
        x="horas",
        y="horario",
        color="periodo",
        barmode="group",
        orientation="h",
        text="horas",
        color_discrete_map={"1ª ordem + ESTENDIDA": CORES["p1"], "2ª ordem + ESTENDIDA": CORES["p2"]},
        category_orders={"periodo": ORDEM_PERIODOS},
        labels={
            "horario": "Horário",
            "horas": "Horas",
            "periodo": "Período operacional",
        },
    )
    fig_horario.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
    estilo_fig(fig_horario, "Top horários por carga — P1 (1ª + ESTENDIDA) × P2 (2ª + ESTENDIDA)", 560).show()

    # ------------------------------------------------------------------
    # Mapa de calor por período + horário contra dia da semana.
    # ------------------------------------------------------------------
    heat = (
        registro_periodos.groupby(["periodo", "dia_semana", "horario"], dropna=False)
        .agg(horas=("horas_periodo", "sum"))
        .reset_index()
    )
    heat["periodo"] = pd.Categorical(heat["periodo"], ORDEM_PERIODOS, ordered=True)
    heat["dia_semana"] = pd.Categorical(heat["dia_semana"], ORDEM_DIAS, ordered=True)
    heat = heat.sort_values(["periodo", "horario", "dia_semana"])
    heat["periodo_horario"] = heat["periodo"].astype(str) + " | " + heat["horario"].astype(str)
    heat_pivot = heat.pivot_table(index="periodo_horario", columns="dia_semana", values="horas", aggfunc="sum", fill_value=0)
    heat_pivot = heat_pivot[[c for c in ORDEM_DIAS if c in heat_pivot.columns]]

    # v14 — remove automaticamente dias sem aula para aproveitar melhor a área visual.
    # Exemplo: se Sexta, Sábado e Domingo somam 0h, eles não aparecem na plotagem.
    dias_com_aula = [c for c in heat_pivot.columns if pd.to_numeric(heat_pivot[c], errors="coerce").fillna(0).sum() > 0]
    heat_pivot_plot = heat_pivot[dias_com_aula] if dias_com_aula else heat_pivot

    # Escala com realce de potência: sem isso, horas entre 2 e 8 saem quase no
    # mesmo tom e o mapa deixa de comunicar concentração.
    _heat_vmin, _heat_vmax = limites_heatmap(heat_pivot_plot.values)
    fig_heat = px.imshow(
        heat_pivot_plot,
        text_auto=True,
        aspect="auto",
        color_continuous_scale=escala_heatmap_gamma(HEATMAP_PALETAS["horas"]),
        zmin=_heat_vmin,
        zmax=_heat_vmax,
        labels=dict(x="Dia da semana", y="Período | Horário", color="Horas"),
    )
    fig_heat.update_traces(textfont=dict(size=13))
    dias_removidos = [c for c in heat_pivot.columns if c not in dias_com_aula]
    titulo_heat = "Mapa de calor — período/horário × dia da semana"
    if dias_removidos:
        titulo_heat += " | dias sem aula ocultados"
    estilo_fig(fig_heat, titulo_heat, 620).show()

    # ------------------------------------------------------------------
    # Distribuição por cargo, também aberta por período.
    # ------------------------------------------------------------------
    dist_cargo = (
        registro_periodos.groupby(["periodo", "cargo"], dropna=False)
        .agg(sessoes=("chave_sessao", "nunique"), horas=("horas_periodo", "sum"), docentes=("professor_key", "nunique"))
        .reset_index()
        .sort_values(["periodo", "horas"], ascending=[True, False])
    )
    dist_cargo_total = (
        registro_periodos.groupby("cargo", dropna=False)
        .agg(sessoes=("chave_sessao", "nunique"), horas=("horas_periodo", "sum"), docentes=("professor_key", "nunique"))
        .reset_index()
        .sort_values("horas", ascending=False)
    )

    fig_cargo_periodo = px.bar(
        dist_cargo.head(30),
        x="horas",
        y="cargo",
        color="periodo",
        orientation="h",
        barmode="group",
        text="horas",
        color_discrete_map={"1ª ordem + ESTENDIDA": CORES["p1"], "2ª ordem + ESTENDIDA": CORES["p2"]},
        labels={"horas":"Horas", "cargo":"Cargo", "periodo":"Período operacional"},
    )
    fig_cargo_periodo.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
    estilo_fig(fig_cargo_periodo, "Carga realizada por cargo — aberta por período", 620).show()

    display(dist_cargo_total)
else:
    display(Markdown("Não foi possível construir estatísticas detalhadas por período porque a planilha Registro bruta não foi localizada."))

# ============================================================
# 8. Estatísticas descritivas por docente
# ============================================================
exibir_titulo("2. Ranking e estatísticas descritivas de docentes", "Carga, dispersão, utilização e balanceamento entre docentes.")

if not aloc_doc.empty:
    ranking_doc = aloc_doc.copy()
    ranking_doc["docente_cargo"] = ranking_doc["professor"].map(nome_exibicao) + " — " + ranking_doc["cargo_final"].map(nome_exibicao)
    ranking_doc = ranking_doc.sort_values("horas_total", ascending=False)

    stats_doc = ranking_doc["horas_total"].describe(percentiles=[.25,.5,.75,.9,.95]).to_frame("horas_total").reset_index()
    display(stats_doc)

    fig_top_doc = px.bar(
        ranking_doc.head(20).sort_values("horas_total"),
        x="horas_total",
        y="docente_cargo",
        orientation="h",
        text="horas_total",
        color="utilizacao_max_%",
        color_continuous_scale=["#BBF7D0", "#FDE68A", "#F97316", "#DC2626"],
        labels={"horas_total":"Horas alocadas (P1+P2)", "docente_cargo":"Docente — cargo", "utilizacao_max_%":"Utilização máx. %"},
    )
    fig_top_doc.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
    estilo_fig(fig_top_doc, "Top 20 docentes por horas alocadas", 750).show()

    # Índice de concentração simples
    total_horas_doc = ranking_doc["horas_total"].sum()
    top_20_pct_n = max(1, math.ceil(len(ranking_doc) * 0.2))
    top_20_horas = ranking_doc.head(top_20_pct_n)["horas_total"].sum()
    indice_concentracao = (top_20_horas / total_horas_doc * 100) if total_horas_doc else 0

    cards_conc = [
        {"titulo":"Docentes com alocação", "valor": inteiro_br(len(ranking_doc)), "subtitulo":"professores no Registro", "cor": CORES["realizado"]},
        {"titulo":"Top 20% concentram", "valor": percentual_br(indice_concentracao, 1), "subtitulo":"das horas alocadas", "cor": CORES["alerta"]},
        {"titulo":"Média por docente", "valor": inteiro_br(ranking_doc["horas_total"].mean(), "h"), "subtitulo":"P1 + P2", "cor": CORES["planejado"]},
        {"titulo":"Mediana por docente", "valor": inteiro_br(ranking_doc["horas_total"].median(), "h"), "subtitulo":"P1 + P2", "cor": CORES["neutro"]},
    ]
    exibir_cards(cards_conc, colunas=4)
else:
    display(Markdown("Sem base docente detalhada para ranking."))

In [ ]:
# ============================================================
# 9. Pareto 80/20 — docentes, cursos, dias, horários e RPA
# ============================================================
exibir_titulo("3. Pareto 80/20", "Identificação das dimensões que concentram a maior parte da carga, demanda ou custo.")

def pareto(df, grupo, valor, nome_valor="valor"):
    if df.empty or grupo not in df.columns or valor not in df.columns:
        return pd.DataFrame()
    tab = (
        df.groupby(grupo, dropna=False)[valor]
        .sum()
        .reset_index()
        .sort_values(valor, ascending=False)
    )
    total = tab[valor].sum()
    tab["participacao_%"] = np.where(total > 0, tab[valor] / total * 100, 0)
    tab["participacao_acumulada_%"] = tab["participacao_%"].cumsum()
    tab["ate_80_%"] = tab["participacao_acumulada_%"] <= 80
    tab = tab.rename(columns={valor: nome_valor})
    return tab

pareto_doc = pareto(ranking_doc if 'ranking_doc' in globals() else pd.DataFrame(), "docente_cargo", "horas_total", "horas")
# Para horas, o Pareto também respeita a abertura por período operacional.
pareto_dia = pareto(dist_dia if 'dist_dia' in globals() else pd.DataFrame(), "dia_periodo", "horas", "horas")
pareto_horario = pareto(dist_horario if 'dist_horario' in globals() else pd.DataFrame(), "horario_periodo", "horas", "horas")

if not mapa_ao_vivo.empty:
    mapa_ao = mapa_ao_vivo.copy()
    mapa_ao["horas_base"] = 2
    pareto_curso = pareto(mapa_ao, "curso_nome", "horas_base", "horas_planejadas")
else:
    pareto_curso = pd.DataFrame()

if not rpa_registro.empty and "valor_rpa_registro" in rpa_registro.columns:
    rpa_reg = rpa_registro.copy()
    rpa_reg["valor_rpa_registro"] = pd.to_numeric(rpa_reg["valor_rpa_registro"], errors="coerce").fillna(0)
    chave_rpa = "professor_realizado" if "professor_realizado" in rpa_reg.columns else "disciplina"
    pareto_rpa = pareto(rpa_reg, chave_rpa, "valor_rpa_registro", "valor_rpa")
else:
    pareto_rpa = pd.DataFrame()

if not pareto_doc.empty:
    qtd_80_doc = int((pareto_doc["participacao_acumulada_%"] <= 80).sum())
    if qtd_80_doc == 0 and len(pareto_doc) > 0:
        qtd_80_doc = 1
    display(Markdown(f"**Pareto docente:** {qtd_80_doc} docentes concentram aproximadamente 80% da carga realizada."))

    fig_pareto_doc = go.Figure()
    tmp = pareto_doc.head(25)
    fig_pareto_doc.add_bar(x=tmp["docente_cargo"], y=tmp["horas"], name="Horas", marker_color=CORES["realizado"])
    fig_pareto_doc.add_scatter(x=tmp["docente_cargo"], y=tmp["participacao_acumulada_%"], name="% acumulado", yaxis="y2", mode="lines+markers", line=dict(color=CORES["critico"], width=3))
    fig_pareto_doc.update_layout(yaxis=dict(title="Horas"), yaxis2=dict(title="% acumulado", overlaying="y", side="right", range=[0,105]))
    fig_pareto_doc.update_xaxes(tickangle=-45)
    estilo_fig(fig_pareto_doc, "Pareto de carga por docente", 650).show()

if 'curso_semanal_rank_pr' in globals() and not curso_semanal_rank_pr.empty:
    display(Markdown("### Cursos com maior demanda semanal — Planejado × Realizado"))
    display(curso_semanal_rank_pr[[
        "periodo", "composicao_periodo", "curso_nome",
        "demanda_semanal_planejada_horas", "demanda_semanal_realizada_horas",
        "delta_semanal_horas", "aderencia_%"
    ]].head(20))
elif not pareto_curso.empty:
    display(Markdown("### Pareto por curso - apenas planejado disponível"))
    display(pareto_curso.head(20))

In [ ]:
# ============================================================
# 10. Sazonalidade operacional P1 × P2
# ============================================================
exibir_titulo("4. Sazonalidade do módulo", "Comparação estatística entre as primeiras 5 semanas e as últimas 5 semanas.")

if not horas_comparativo.empty:
    saz = horas_comparativo.copy()
    saz["periodo"] = pd.Categorical(saz["periodo"], ORDEM_PERIODOS, ordered=True)
    saz = saz.sort_values("periodo")

    fig_saz = go.Figure()
    fig_saz.add_bar(x=saz["periodo"], y=saz["horas_planejado"], name="Planejado", marker_color=CORES["planejado"], text=saz["horas_planejado"])
    fig_saz.add_bar(x=saz["periodo"], y=saz["horas_realizado"], name="Realizado", marker_color=CORES["realizado"], text=saz["horas_realizado"])
    fig_saz.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
    fig_saz.update_layout(barmode="group", yaxis_title="Horas")
    estilo_fig(fig_saz, "Planejado × realizado por período", 500).show()

    if len(saz) >= 2:
        p1 = saz.loc[saz["periodo"].astype(str).eq("1ª ordem + ESTENDIDA"), "horas_planejado"].sum()
        p2 = saz.loc[saz["periodo"].astype(str).eq("2ª ordem + ESTENDIDA"), "horas_planejado"].sum()
        variacao = ((p2 - p1) / p1 * 100) if p1 else 0
        display(Markdown(f"**Leitura executiva:** o 2ª ordem + ESTENDIDA possui demanda planejada {percentual_br(variacao,1)} em relação ao 1ª ordem + ESTENDIDA."))

if not registro_periodos.empty:
    presenca_doc = registro_periodos.groupby(["professor_key","professor","periodo"]).size().reset_index(name="qtd")
    pvt = presenca_doc.pivot_table(index=["professor_key","professor"], columns="periodo", values="qtd", fill_value=0).reset_index()
    pvt["perfil_periodo"] = np.select(
        [
            (pvt.get("1ª ordem + ESTENDIDA",0) > 0) & (pvt.get("2ª ordem + ESTENDIDA",0) > 0),
            (pvt.get("1ª ordem + ESTENDIDA",0) > 0) & (pvt.get("2ª ordem + ESTENDIDA",0) == 0),
            (pvt.get("1ª ordem + ESTENDIDA",0) == 0) & (pvt.get("2ª ordem + ESTENDIDA",0) > 0),
        ],
        ["Atua nos dois períodos", "Somente P1", "Somente P2"],
        default="Sem classificação"
    )
    perfil_periodo = pvt.groupby("perfil_periodo").size().reset_index(name="docentes")
    fig_perfil = px.pie(perfil_periodo, names="perfil_periodo", values="docentes", hole=.45,
                        color_discrete_sequence=[CORES["p1"], CORES["p2"], CORES["estendida"], CORES["neutro"]])
    estilo_fig(fig_perfil, "Perfil de atuação dos docentes por período", 460).show()

In [ ]:
# ============================================================
# 11. Eficiência, capacidade e risco docente
# ============================================================
exibir_titulo("5. Eficiência, capacidade e risco docente", "Uso da CH letiva alocável, saldo operacional e classificação de risco.")

if not delta_docente.empty:
    delta_show = delta_docente.copy()
    display(delta_show)

    fig_delta = go.Figure()
    fig_delta.add_bar(x=delta_show["periodo"], y=delta_show["demanda_planejada_horas"], name="Demanda planejada", marker_color=CORES["planejado"], text=delta_show["demanda_planejada_horas"])
    fig_delta.add_bar(x=delta_show["periodo"], y=delta_show["ch_letiva_alocavel_horas"], name="CH letiva alocável", marker_color=CORES["capacidade"], text=delta_show["ch_letiva_alocavel_horas"])
    if "horas_alocadas_registro" in delta_show.columns:
        fig_delta.add_bar(x=delta_show["periodo"], y=delta_show["horas_alocadas_registro"], name="Alocado no Registro", marker_color=CORES["realizado"], text=delta_show["horas_alocadas_registro"])
    fig_delta.update_traces(texttemplate="%{text:.0f}h", textposition="outside")
    fig_delta.update_layout(barmode="group", yaxis_title="Horas")
    estilo_fig(fig_delta, "Demanda planejada × CH letiva alocável × alocação", 520).show()

# Risco docente
if not aloc_doc.empty:
    risco = aloc_doc.copy()
    risco["score_risco"] = 0.0

    # Métricas de concentração por dia
    if not registro_periodos.empty:
        dia_doc = registro_periodos.groupby(["professor_key","dia_semana"])["horas_periodo"].sum().reset_index()
        max_dia = dia_doc.groupby("professor_key")["horas_periodo"].max().reset_index(name="max_horas_mesmo_dia")
        total_doc = registro_periodos.groupby("professor_key")["horas_periodo"].sum().reset_index(name="total_horas_periodo")
        conc = max_dia.merge(total_doc, on="professor_key", how="left")
        conc["concentracao_dia_%"] = np.where(conc["total_horas_periodo"]>0, conc["max_horas_mesmo_dia"]/conc["total_horas_periodo"]*100, 0)
        risco = risco.merge(conc[["professor_key","concentracao_dia_%"]], on="professor_key", how="left")
    else:
        risco["concentracao_dia_%"] = 0

    risco["acima_capacidade"] = risco["utilizacao_max_%"].fillna(0) > 100
    risco.loc[risco["utilizacao_max_%"].fillna(0) >= 90, "score_risco"] += 25
    risco.loc[risco["utilizacao_max_%"].fillna(0) >= 100, "score_risco"] += 20
    risco.loc[risco["concentracao_dia_%"].fillna(0) >= 60, "score_risco"] += 15
    risco.loc[risco["docente_ativo_base"].fillna(False) == False, "score_risco"] += 25

    p90_horas = risco["horas_total"].quantile(.90) if risco["horas_total"].notna().any() else 0
    risco.loc[risco["horas_total"].fillna(0) >= p90_horas, "score_risco"] += 10

    p75_rpa = risco["valor_rpa"].quantile(.75) if risco["valor_rpa"].fillna(0).gt(0).any() else 0
    if p75_rpa > 0:
        risco.loc[risco["valor_rpa"].fillna(0) >= p75_rpa, "score_risco"] += 10

    risco["score_risco"] = risco["score_risco"].clip(0,100)
    risco["classe_risco"] = pd.cut(
        risco["score_risco"],
        bins=[-1, 29, 59, 79, 100],
        labels=["Baixo", "Médio", "Alto", "Crítico"]
    )
    risco["docente_cargo"] = risco["professor"].map(nome_exibicao) + " — " + risco["cargo_final"].map(nome_exibicao)
    risco = risco.sort_values(["score_risco","horas_total"], ascending=False)

    fig_risco = px.bar(
        risco.head(25).sort_values("score_risco"),
        x="score_risco",
        y="docente_cargo",
        orientation="h",
        color="classe_risco",
        text="score_risco",
        color_discrete_map={"Baixo":"#22C55E","Médio":"#F59E0B","Alto":"#F97316","Crítico":"#DC2626"},
        labels={"score_risco":"Score de risco", "docente_cargo":"Docente — cargo", "classe_risco":"Risco"},
    )
    fig_risco.update_traces(texttemplate="%{text:.0f}", textposition="outside")
    estilo_fig(fig_risco, "Top docentes por risco operacional", 760).show()

    display(risco[["professor","cargo_final","horas_total","utilizacao_p1_%","utilizacao_p2_%","concentracao_dia_%","docente_ativo_base","valor_rpa","score_risco","classe_risco"]].head(30))

In [ ]:
# ============================================================
# 12. Outliers estatísticos
# ============================================================
exibir_titulo("6. Outliers estatísticos", "Detecção por intervalo interquartil e z-score simples.")

def detectar_outliers(df, valor_col, contexto_col=None, nome=""):
    if df.empty or valor_col not in df.columns:
        return pd.DataFrame()
    out = df.copy()
    out[valor_col] = pd.to_numeric(out[valor_col], errors="coerce")
    vals = out[valor_col].dropna()
    if vals.empty:
        return pd.DataFrame()
    q1 = vals.quantile(.25)
    q3 = vals.quantile(.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    media = vals.mean()
    desvio = vals.std(ddof=0)
    out["z_score"] = np.where(desvio > 0, (out[valor_col] - media) / desvio, 0)
    out["outlier_iqr"] = (out[valor_col] < lim_inf) | (out[valor_col] > lim_sup)
    out["outlier_zscore"] = out["z_score"].abs() >= 2
    out["tipo_analise"] = nome
    cols = ([contexto_col] if contexto_col else []) + [valor_col, "z_score", "outlier_iqr", "outlier_zscore", "tipo_analise"]
    return out[out["outlier_iqr"] | out["outlier_zscore"]][cols].sort_values(valor_col, ascending=False)

out_doc = detectar_outliers(ranking_doc if 'ranking_doc' in globals() else pd.DataFrame(), "horas_total", "docente_cargo", "Carga docente")
out_curso = detectar_outliers(pareto_curso if 'pareto_curso' in globals() else pd.DataFrame(), "horas_planejadas", "curso_nome", "Demanda por curso")
out_rpa = detectar_outliers(pareto_rpa if 'pareto_rpa' in globals() else pd.DataFrame(), "valor_rpa", pareto_rpa.columns[0] if 'pareto_rpa' in globals() and not pareto_rpa.empty else None, "Valor RPA")

display(Markdown(f"""
- Outliers de carga docente: **{len(out_doc)}**
- Outliers de demanda por curso: **{len(out_curso)}**
- Outliers de valor RPA: **{len(out_rpa)}**
"""))

if not out_doc.empty:
    display(out_doc.head(20))
if not out_rpa.empty:
    display(out_rpa.head(20))

In [ ]:
# ============================================================
# 13. Simulação de cenários
# ============================================================
exibir_titulo("7. Simulação de cenários", "Testes de crescimento de demanda e leitura de capacidade disponível.")

cenarios = []
if not delta_docente.empty:
    for _, r in delta_docente.iterrows():
        for crescimento in [0, 5, 10, 15, 20, 30]:
            demanda = num(r.get("demanda_planejada_horas")) * (1 + crescimento/100)
            capacidade = num(r.get("ch_letiva_alocavel_horas"))
            alocado = num(r.get("horas_alocadas_registro"))
            cenarios.append({
                "periodo": r["periodo"],
                "crescimento_demanda_%": crescimento,
                "demanda_simulada_horas": demanda,
                "ch_letiva_alocavel_horas": capacidade,
                "saldo_demanda_vs_capacidade": capacidade - demanda,
                "utilizacao_demanda_%": (demanda / capacidade * 100) if capacidade else np.nan,
                "utilizacao_alocada_atual_%": (alocado / capacidade * 100) if capacidade else np.nan,
                "status_cenario": "Comporta" if capacidade >= demanda else "Déficit",
            })
cenarios_df = pd.DataFrame(cenarios)

if not cenarios_df.empty:
    fig_cen = px.line(
        cenarios_df,
        x="crescimento_demanda_%",
        y="utilizacao_demanda_%",
        color="periodo",
        markers=True,
        color_discrete_map={"1ª ordem + ESTENDIDA":CORES["p1"], "2ª ordem + ESTENDIDA":CORES["p2"]},
        labels={"crescimento_demanda_%":"Crescimento da demanda (%)", "utilizacao_demanda_%":"Utilização da CH alocável (%)"},
    )
    fig_cen.add_hline(y=100, line_dash="dash", line_color=CORES["critico"], annotation_text="Limite de capacidade")
    estilo_fig(fig_cen, "Cenário — demanda simulada versus capacidade docente", 520).show()
    display(cenarios_df)

# Oportunidade de internalização de RPA
if not rpa_registro.empty:
    rpa_total = len(rpa_registro)
    rpa_horas_estimadas = rpa_total * 2
    rpa_valor = pd.to_numeric(rpa_registro.get("valor_rpa_registro", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()
else:
    rpa_total, rpa_horas_estimadas, rpa_valor = 0, 0, 0

if not delta_docente.empty:
    capacidade_sobra_min = (delta_docente["ch_letiva_alocavel_horas"] - delta_docente["horas_alocadas_registro"]).min()
else:
    capacidade_sobra_min = 0
absorvivel = max(0, min(rpa_horas_estimadas, capacidade_sobra_min))
valor_medio_hora_rpa = (rpa_valor / rpa_horas_estimadas) if rpa_horas_estimadas else 0
economia_potencial = absorvivel * valor_medio_hora_rpa

cards_cenario = [
    {"titulo":"RPAs no Registro", "valor": inteiro_br(rpa_total), "subtitulo":"com ou sem valor", "cor": CORES["rpa"]},
    {"titulo":"Horas RPA estimadas", "valor": inteiro_br(rpa_horas_estimadas, "h"), "subtitulo":"qtd RPA × 2h", "cor": CORES["rpa"]},
    {"titulo":"Absorção teórica", "valor": inteiro_br(absorvivel, "h"), "subtitulo":"limitada pela menor sobra P1/P2", "cor": CORES["realizado"]},
    {"titulo":"Economia potencial", "valor": moeda_br(economia_potencial), "subtitulo":"estimativa por valor médio/hora RPA", "cor": CORES["horas"]},
]
exibir_cards(cards_cenario, colunas=4)

In [ ]:
# ============================================================
# 14. Histórico automático entre módulos
# ============================================================
exibir_titulo("8. Histórico automático entre módulos", "Criação de uma base acumulada para acompanhar evolução entre módulos.")

hist_path = OUTPUT_DIR / "historico_siap_inteligencia.csv"

hist_row = {
    "modulo": MODULO,
    "ano": ANO,
    "data_execucao": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "qtd_planejado": num(linha_resumo.get("qtd_planejado")),
    "qtd_realizado": num(linha_resumo.get("qtd_realizado")),
    "conformidade_%": num(linha_resumo.get("conformidade_%")),
    "valor_rpa_mapa": num(linha_resumo.get("valor_rpa_mapa")),
    "valor_rpa_registro": num(linha_resumo.get("valor_rpa_registro")),
    "rpa_registro_total": num(linha_resumo.get("rpa_registro_total")),
    "ch_letiva_ativa_total": num(linha_resumo.get("ch_letiva_ativa_total")),
    "ch_letiva_alocavel_total": num(linha_resumo.get("ch_letiva_alocavel_total")),
    "horas_p1_realizadas": horas_realizado.loc[horas_realizado["periodo"].eq("1ª ordem + ESTENDIDA"), "horas"].sum() if not horas_realizado.empty else np.nan,
    "horas_p2_realizadas": horas_realizado.loc[horas_realizado["periodo"].eq("2ª ordem + ESTENDIDA"), "horas"].sum() if not horas_realizado.empty else np.nan,
    "docentes_ativos": num(linha_resumo.get("professores_unicos_ativos_parametros")),
}

if ATUALIZAR_HISTORICO:
    if hist_path.exists():
        hist = pd.read_csv(hist_path)
    else:
        hist = pd.DataFrame()
    # Remove execução anterior do mesmo módulo/ano para evitar duplicidade em reprocessamento.
    if not hist.empty:
        hist = hist[~((hist["modulo"].astype(str) == str(MODULO)) & (hist["ano"].astype(str) == str(ANO)))]
    hist = pd.concat([hist, pd.DataFrame([hist_row])], ignore_index=True)
    hist = hist.sort_values(["ano","modulo"])
    try:
        hist.to_csv(hist_path, index=False, encoding="utf-8-sig")
    except PermissionError:
        # Em ambientes corporativos ou pastas sincronizadas, o arquivo histórico pode estar bloqueado.
        # Nesse caso, gera uma cópia timestampada sem interromper o notebook.
        hist_path_alt = OUTPUT_DIR / f"historico_siap_inteligencia_{MODULO}_{timestamp}.csv"
        hist.to_csv(hist_path_alt, index=False, encoding="utf-8-sig")
        print(f"Histórico principal bloqueado. Cópia gerada em: {hist_path_alt}")
else:
    hist = pd.DataFrame([hist_row])

display(hist)

if len(hist) >= 2:
    fig_hist = px.line(
        hist,
        x="modulo",
        y=["qtd_planejado","qtd_realizado"],
        markers=True,
        labels={"value":"Quantidade", "modulo":"Módulo", "variable":"Métrica"},
    )
    estilo_fig(fig_hist, "Histórico — planejado versus realizado", 500).show()

    fig_hist_rpa = px.line(
        hist,
        x="modulo",
        y="valor_rpa_registro",
        markers=True,
        labels={"valor_rpa_registro":"Valor RPA registrado", "modulo":"Módulo"},
    )
    estilo_fig(fig_hist_rpa, "Histórico — custo RPA registrado", 500).show()
else:
    display(Markdown("A base histórica foi iniciada. A partir do próximo módulo, o notebook passará a exibir evolução e tendências."))

In [ ]:
# ============================================================
# 15. Geração automática de insights
# ============================================================
exibir_titulo("9. Insights automáticos", "Texto analítico gerado a partir dos indicadores estatísticos.")

insights = []
insights.append("Todas as análises de horas deste notebook são lidas por período operacional e, sempre que a dimensão existe nas duas bases, com contraponto Planejado × Realizado.")

if not horas_comparativo.empty:
    p1_plan = horas_comparativo.loc[horas_comparativo["periodo"].eq("1ª ordem + ESTENDIDA"), "horas_planejado"].sum()
    p2_plan = horas_comparativo.loc[horas_comparativo["periodo"].eq("2ª ordem + ESTENDIDA"), "horas_planejado"].sum()
    if p1_plan:
        insights.append(f"O 2ª ordem + ESTENDIDA possui demanda planejada {percentual_br((p2_plan-p1_plan)/p1_plan*100, 1)} maior que o 1ª ordem + ESTENDIDA.")

if 'indice_concentracao' in globals():
    insights.append(f"O top 20% dos docentes concentra {percentual_br(indice_concentracao, 1)} das horas realizadas, indicando o nível de concentração operacional da carga.")

if not delta_docente.empty:
    cap = delta_docente["ch_letiva_alocavel_horas"].iloc[0]
    util_p1 = delta_docente.loc[delta_docente["periodo"].eq("1ª ordem + ESTENDIDA"), "utilizacao_realizada_%"].sum()
    util_p2 = delta_docente.loc[delta_docente["periodo"].eq("2ª ordem + ESTENDIDA"), "utilizacao_realizada_%"].sum()
    insights.append(f"A CH letiva alocável ativa é de {inteiro_br(cap, 'h')} por período. A utilização realizada foi {percentual_br(util_p1,1)} no P1 e {percentual_br(util_p2,1)} no P2.")

if 'risco' in globals() and not risco.empty:
    criticos = int((risco["classe_risco"].astype(str) == "Crítico").sum())
    altos = int((risco["classe_risco"].astype(str) == "Alto").sum())
    insights.append(f"A análise de risco docente identificou {criticos} docentes em risco crítico e {altos} em risco alto, considerando utilização, concentração e governança cadastral.")

if 'curso_semanal_rank_pr' in globals() and not curso_semanal_rank_pr.empty:
    top_curso = curso_semanal_rank_pr.iloc[0]
    insights.append(
        f"O curso com maior demanda semanal AO VIVO é {top_curso['curso_nome']} no {top_curso['periodo']} "
        f"({top_curso['composicao_periodo']}): {inteiro_br(top_curso['demanda_semanal_planejada_horas'], 'h/sem')} planejadas "
        f"e {inteiro_br(top_curso['demanda_semanal_realizada_horas'], 'h/sem')} realizadas."
    )
elif not pareto_curso.empty:
    top_curso = pareto_curso.iloc[0]
    insights.append(f"O curso com maior demanda planejada é {top_curso['curso_nome']}, com {inteiro_br(top_curso['horas_planejadas'], 'h')} planejadas em aulas AO VIVO.")

if rpa_valor > 0:
    insights.append(f"O custo RPA registrado soma {moeda_br(rpa_valor)}; a análise de cenário estima economia potencial de até {moeda_br(economia_potencial)} se houver internalização operacionalmente viável.")

for i, txt in enumerate(insights, 1):
    display(Markdown(f"**Insight {i}.** {txt}"))

insights_df = pd.DataFrame({"insight": insights})

In [ ]:
# ============================================================
# 16. Dashboard HTML e exportações
# ============================================================
exibir_titulo("10. Exportações", "Geração de Excel analítico e Dashboard HTML.")

# Coleta de figuras para HTML
figs_html = []
for name in ["fig_ordem_pr","fig_dia_pr","fig_horario_pr","fig_curso_pr","fig_dia","fig_horario","fig_heat","fig_cargo_periodo","fig_top_doc","fig_pareto_doc","fig_saz","fig_perfil","fig_delta","fig_risco","fig_cen"]:
    if name in globals():
        try:
            figs_html.append((name, salvar_fig_html(globals()[name])))
        except Exception:
            pass

excel_out = OUTPUT_DIR / f"SGA_Inteligencia_Estatistica_Modulo_{MODULO}_{timestamp}.xlsx"
html_out = OUTPUT_DIR / f"SGA_Inteligencia_Dashboard_Modulo_{MODULO}_{timestamp}.html"

abas_export = {
    "README": pd.DataFrame({
        "Descrição": [
            "Notebook complementar de Inteligência Estatística e Histórica.",
            "Base primária: Excel consolidado SGA.",
            "Base detalhada: Registro PREENCHIMENTO|LETIVAS, quando localizado.",
            "Linhas totalmente iguais do Registro são identificadas para governança, mas permanecem nos cálculos.",
            "As sessões realizadas consideram todas as linhas válidas do Registro; duplicidades são sinalizadas para governança.",
            "ESTENDIDA é considerada no 1ª ordem + ESTENDIDA e no 2ª ordem + ESTENDIDA.",
            "Cursos: demanda semanal P1 = (1ª + ESTENDIDA) × 2h/semana; P2 = (2ª + ESTENDIDA) × 2h/semana."
        ]
    }),
    "Resumo Executivo": pd.DataFrame([hist_row]),
    "Insights": insights_df,
    "PR Ordem": comp_ordem_pr if 'comp_ordem_pr' in globals() else pd.DataFrame(),
    "PR Dia Periodo": comp_dia_periodo_pr if 'comp_dia_periodo_pr' in globals() else pd.DataFrame(),
    "PR Horario Periodo": comp_horario_periodo_pr if 'comp_horario_periodo_pr' in globals() else pd.DataFrame(),
    "PR Curso Semanal": curso_semanal_rank_pr if 'curso_semanal_rank_pr' in globals() else pd.DataFrame(),
    "PR Curso Periodo": curso_semanal_periodo_pr if 'curso_semanal_periodo_pr' in globals() else pd.DataFrame(),
    "PR Curso Total Apoio": comp_curso_total_pr if 'comp_curso_total_pr' in globals() else pd.DataFrame(),
    "Distribuição Dia Periodo": dist_dia if 'dist_dia' in globals() else pd.DataFrame(),
    "Distribuição Horário P": dist_horario if 'dist_horario' in globals() else pd.DataFrame(),
    "Heatmap Dias Plot": heat_pivot_plot.reset_index() if 'heat_pivot_plot' in globals() else pd.DataFrame(),
    "Distribuição Cargo P": dist_cargo if 'dist_cargo' in globals() else pd.DataFrame(),
    "Distribuição Cargo Total": dist_cargo_total if 'dist_cargo_total' in globals() else pd.DataFrame(),
    "Ranking Docentes": ranking_doc if 'ranking_doc' in globals() else pd.DataFrame(),
    "Pareto Docentes": pareto_doc,
    "Pareto Cursos": pareto_curso,
    "Pareto Dia": pareto_dia,
    "Pareto Horário": pareto_horario,
    "Pareto RPA": pareto_rpa,
    "Sazonalidade": saz if 'saz' in globals() else pd.DataFrame(),
    "Perfil Periodo Docentes": perfil_periodo if 'perfil_periodo' in globals() else pd.DataFrame(),
    "Risco Docente": risco if 'risco' in globals() else pd.DataFrame(),
    "Outliers Docentes": out_doc,
    "Outliers Cursos": out_curso,
    "Outliers RPA": out_rpa,
    "Cenarios": cenarios_df,
    "Historico": hist,
    "Check Consistencia": check,
    "Registro Sessoes": registro_sessoes,
    "Registro Periodos": registro_periodos,
    "Registro Dup Exatas": registro_duplicatas_exatas,
    "Registro Dup Removidas": registro_duplicatas_removidas,
}

try:
    with pd.ExcelWriter(excel_out, engine="xlsxwriter") as writer:
        for nome, df_export in abas_export.items():
            safe_name = nome[:31]
            df_export.to_excel(writer, sheet_name=safe_name, index=False)
            ws = writer.sheets[safe_name]
            wb = writer.book
            header_fmt = wb.add_format({"bold": True, "bg_color": "#0F172A", "font_color": "white", "border": 1})
            for col_num, value in enumerate(df_export.columns):
                ws.write(0, col_num, value, header_fmt)
                width = min(max(12, len(str(value)) + 4), 45)
                ws.set_column(col_num, col_num, width)
except Exception as e:
    print("Falha com xlsxwriter, tentando openpyxl:", e)
    with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
        for nome, df_export in abas_export.items():
            df_export.to_excel(writer, sheet_name=nome[:31], index=False)

# HTML Premium
css = f"""
<style>
body {{
    margin:0; background:#F8FAFC; color:#0F172A; font-family:Inter, Arial, sans-serif;
}}
.hero {{
    padding:38px 46px; background:linear-gradient(135deg,#0F172A,#1E3A8A);
    color:white;
}}
.hero h1 {{ margin:0; font-size:34px; }}
.hero p {{ color:#CBD5E1; font-size:16px; max-width:1000px; }}
.container {{ max-width:1220px; margin:0 auto; padding:28px 24px 60px 24px; }}
.grid {{ display:grid; grid-template-columns:repeat(4,1fr); gap:16px; margin:22px 0; }}
.card {{
    background:white; border:1px solid #E2E8F0; border-radius:18px; padding:20px;
    box-shadow:0 10px 25px rgba(15,23,42,.07);
}}
.card .label {{ color:#64748B; text-transform:uppercase; font-size:12px; font-weight:800; letter-spacing:.06em; }}
.card .value {{ font-size:30px; font-weight:900; margin-top:8px; }}
.section {{
    margin-top:34px; padding:24px; background:white; border:1px solid #E2E8F0; border-radius:20px;
    box-shadow:0 10px 25px rgba(15,23,42,.06);
}}
.section h2 {{ margin:0 0 8px 0; font-size:24px; }}
.muted {{ color:#64748B; }}
.insight {{ padding:12px 14px; margin:10px 0; border-left:5px solid {CORES['horas']}; background:#FFFBEB; border-radius:10px; }}
.tbl {{ border-collapse:collapse; width:100%; font-size:13px; }}
.tbl th {{ background:#0F172A; color:white; padding:9px; text-align:left; }}
.tbl td {{ border-bottom:1px solid #E2E8F0; padding:8px; }}
.footer {{ color:#64748B; font-size:12px; margin-top:30px; }}
</style>
"""

cards_html = "".join([
    card_html("Módulo", f"{MODULO}/{ANO}", "base analisada", CORES["neutro"]),
    card_html("Sessões realizadas", inteiro_br(len(registro_sessoes)), "Registro integral", CORES["realizado"]),
    card_html("CH alocável", inteiro_br(linha_resumo.get("ch_letiva_alocavel_total",0), "h"), "por período", CORES["capacidade"]),
    card_html("RPA realizado", moeda_br(linha_resumo.get("valor_rpa_registro",0)), "custo registrado", CORES["rpa"]),
])

insights_html = "".join([f"<div class='insight'><b>Insight {i+1}.</b> {txt}</div>" for i, txt in enumerate(insights)])

fig_sections = ""
titulos_figs = {
    "fig_ordem_pr":"Ordem — Planejado × Realizado",
    "fig_dia_pr":"Dia da semana — Planejado × Realizado por período",
    "fig_horario_pr":"Horário — Planejado × Realizado por período",
    "fig_curso_pr":"Demanda semanal por curso — Planejado × Realizado",
    "fig_dia":"Distribuição realizada por dia da semana — P1 × P2",
    "fig_horario":"Distribuição por horário — P1 × P2",
    "fig_heat":"Mapa de calor operacional por período",
    "fig_cargo_periodo":"Carga por cargo — P1 × P2",
    "fig_top_doc":"Ranking docente",
    "fig_pareto_doc":"Pareto docente",
        "fig_saz":"Sazonalidade P1 × P2",
    "fig_perfil":"Perfil de atuação docente",
    "fig_delta":"Demanda × capacidade × alocação",
    "fig_risco":"Risco docente",
    "fig_cen":"Cenários de demanda",
}
for name, html_fig in figs_html:
    fig_sections += f"<div class='section'><h2>{titulos_figs.get(name,name)}</h2>{html_fig}</div>"

html = f"""
<!DOCTYPE html>
<html lang="pt-br">
<head>
<meta charset="UTF-8">
<title>Inteligência Estatística — Módulo {MODULO} | {RELATORIO_ORGANIZACAO}</title>
<script src="https://cdn.plot.ly/plotly-2.30.0.min.js"></script>
{css}
</head>
<body>
<div class="hero">
    <h1>Inteligência Estatística e Histórica — {RELATORIO_ORGANIZACAO}</h1>
    <p>Módulo {MODULO}/{ANO}. Análise complementar com Previsto × Realizado por dimensões, distribuição operacional, Pareto, risco docente, outliers, cenários e histórico.</p>
    <p><b>{RELATORIO_AUTOR_NOME}</b> — {RELATORIO_AUTOR_CARGO}{(" | " + RELATORIO_AUTOR_AREA) if str(RELATORIO_AUTOR_AREA).strip() else ""} · Atualizado em {carimbo_atualizacao('%d/%m/%Y %H:%M')}</p>
</div>
<div class="container">
    <div class="grid">{cards_html}</div>
    <div class="section">
        <h2>Resumo analítico</h2>
        {insights_html}
    </div>
    {fig_sections}
    <div class="section">
        <h2>Top riscos docentes</h2>
        {tabela_html(risco[['professor','cargo_final','horas_total','utilizacao_p1_%','utilizacao_p2_%','score_risco','classe_risco']].head(15) if 'risco' in globals() and not risco.empty else pd.DataFrame())}
    </div>
    <div class="section">
        <h2>Cenários</h2>
        {tabela_html(cenarios_df.head(20) if not cenarios_df.empty else pd.DataFrame())}
    </div>
    <p class="footer">{assinatura_relatorio()} · Documento atualizado em {carimbo_atualizacao('%d/%m/%Y %H:%M')}.</p>
</div>
</body>
</html>
"""

html_out.write_text(html, encoding="utf-8")

display(Markdown(f"""
### Arquivos gerados

- Excel analítico: `{excel_out}`
- Dashboard HTML: `{html_out}`
- Histórico acumulado: `{hist_path}`
"""))

print("Exportações concluídas com sucesso.")